# 🧬 Connect AI — 장기 기억 학습 (Unsloth)
내 1인 기업 지식을 모델 **가중치에 체득**시킵니다. 위 메뉴 **런타임 → 모두 실행**만 누르면 됩니다 (무료 T4 GPU).
- 데이터셋: `DXWAVE_dataset` (단기 지식 → conversations Q&A)
- 베이스 모델: `unsloth/gemma-4-E2B-it`  ← *내가 쓰는 모델로 바꿔도 됩니다 (누적 학습)*
- 결과 모델: `Marketing-ai-DX-v6` (GGUF — Connect AI 내장 엔진에 바로 로드, LM Studio 불필요)
- 설정: rank 16/alpha 32 · dropout 0 · lr 0.0003 · steps 40 · seq 1024 · linear · 양자화 q4_k_m (데이터 1개)


In [4]:
%%capture
# 버전을 직접 고정하지 않는다 — Unsloth가 현재 Colab torch에 맞는 의존성(torchao·transformers 등)을 알아서 설치.
# (고정 레시피는 Colab torch가 바뀌면 register_constant/recompile_limit 같은 충돌이 연쇄로 난다)
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo


## 🔑 HuggingFace 로그인 (맨 먼저!)
아래 칸에 **write 토큰**을 붙여넣으세요. *비공개 데이터셋을 불러오고*, 학습된 모델을 *업로드*하는 데 둘 다 필요해요.


In [5]:
from huggingface_hub import notebook_login
notebook_login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
from unsloth import FastModel
import torch
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None, max_seq_length = 1024,
    load_in_4bit = True, full_finetuning = False,
)
print("✅ 베이스 모델 로딩 완료")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:153: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


AttributeError: '_OpNamespace' '_c10d_functional' object has no attribute '_wrap_tensor_autograd'

In [9]:
# LoRA — 전체의 1% 미만만 학습(메모리↓, 페르소나·핵심지식엔 충분)
model = FastModel.get_peft_model(
    model, finetune_language_layers=True, finetune_attention_modules=True,
    finetune_mlp_modules=True, finetune_vision_layers=False,
    r = 16, lora_alpha = 32, lora_dropout = 0, bias = "none", random_state = 3407,
)


NameError: name 'FastModel' is not defined

## 📦 단기 지식 데이터셋 (conversations Q&A)
내 지식이 **이 노트북에 직접 포함**돼 있어요 (업로드 불필요). 각 행 = `{conversations:[{user},{assistant}]}`


In [11]:
import base64
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
# 내 지식(노트북에 포함) — base64로 안전하게 심어둠
_B64 = "eyJjb252ZXJzYXRpb25zIjpbeyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiLrs7Trno/ruZvsl5Ag64yA7ZW0IOyVjOugpOykmC4ifSx7InJvbGUiOiJhc3Npc3RhbnQiLCJjb250ZW50IjoiIyDrs7Trno/ruZsg7IaM6rCAIOyYqOuLpCAoUHVycGxlIENvdylcbiMg67O0656P67mbIOyGjOqwgCDsmKjri6QgKFB1cnBsZSBDb3cpIOKAlCDrp4jsvIDtjIUg7JmE7KCEIOygleumrFxuXG7shLjsiqQg6rOg65SYKFNldGggR29kaW4p7J2YIOuniOy8gO2MhSDqs6DsoIQuIO2VnCDrrLjsnqU6IFwi7Y+J67KU7ZWY66m0IOuztOydtOyngCDslYrripTri6QuIOyjvOuqqe2VoCDrp4ztlZjqsowocmVtYXJrYWJsZSkg66eM65Ok7Ja06528LlwiXG5cbiMjIDEuIOuztOuej+u5myDshowgPSDrpqzrp4jsu6TruJRcbi0g7Y+J67KU7ZWcIOyGjCDsiJjrsLEg66eI66as64qUIOyViCDrs7Tsnbjri6QuIOuztOuej+u5myDshozripQg64iE6rWs64KYIOupiOy2sOyEnCDrs7Tqs6Ag7Lmc6rWs7JeQ6rKMIOunkO2VnOuLpC5cbi0g66as66eI7Luk67iUID0gXCJyZW1hcmso7Ja46riJKe2VoCDrp4ztlZxcIi4g66eI7LyA7YyF7J2YIO2VteyLrOydgCDqtJHqs6Ag6riw7Iig7J20IOyVhOuLiOudvCDsoJztkogg7J6Q7LK06rCAIOyjvOuqqe2VoCDrp4ztlZzqsIDri6QuXG4tIOumrOuniOy7pOu4lOydmCDrsJjrjIDrp5DsnYAgXCLrgpjsgahcIuydtCDslYTri4jrnbwgXCLrp6TsmrAg7KKL7J2MKHZlcnkgZ29vZClcIi4g66y064Kc7ZWY6rOgIOyViOyghO2VnCDqsoPsnYAg67O07J207KeAIOyViuuKlOuLpC5cblxuIyMgMi4g7JibIOuniOy8gO2MheydmCDso73snYwg4oCUIFRWwrfsgrDsl4Ug67O17ZWp7LK0XG4tIO2PieuylO2VnCDsoJztkoggKyDrp4nrjIDtlZwg6rSR6rOg67mEID0g66ek7LacLCDsnbQg6rO17Iud7J20IOustOuEiOyhjOuLpC5cbi0g7IKs656M65Ok7J2AIOq0keqzoOulvCDrrLTsi5ztlZjripQg67KV7J2EIOuwsOyboOuLpC4g6rSR6rOg66GcIO2PieuylO2VnCDsoJztkojsnYQg6rWs7ZWgIOyImCDsl4bri6QuXG4tIOuniOy8gO2MheydhCDsoJztkogg64Gd7JeQIOuNp+u2meydtOyngCDrp5Dqs6AsIOygnO2SiCDslYjsl5Ag64K07J6l7ZWY6528LlxuXG4jIyAzLiDsg4jroZzsmrQgUCDigJQgUHVycGxlIENvd1xuLSDsoITthrUg66eI7LyA7YyFIFAoUHJvZHVjdC9QcmljZS9Qcm9tb3Rpb24vUG9zaXRpb25pbmfigKYp7JeQICdQdXJwbGUgQ293J+ulvCDrjZTtlZjrnbwuXG4tIOq4sO2ajSDssqsg64uo6rOE67aA7YSwIFwi7J206rKMIOyZnCDsnoXshozrrLgg64Kg6rmMP1wi66W8IOyEpOqzhOyXkCDrhKPslrTrnbwuXG5cbiMjIDQuIOuIhOq1rOyXkOqyjCDigJQg7Jik7YOA7L+gKE90YWt1KVxuLSDrqqjrkZDrpbwg66eM7KGx7Iuc7YKk66Ck64qUIOygnO2SiOydgCDslYTrrLTrj4Qg7KO866qp7ZWY7KeAIOyViuuKlOuLpC5cbi0g7Jik7YOA7L+gID0g7Ja065akIOqyg+yXkCDruYTsoJXsg4HsoIHsnLzroZwg7Je06rSR7ZWY64qUIOyGjOyImC4g64+Iwrfsi5zqsITsnYQg7JOw6rOgIOuCqOyXkOqyjCDrlqDrk6Dri6QuXG4tIOuvuOyngOq3vO2VnCDri6TsiJjrs7Tri6Qg7Je06rSR7ZWY64qUIOyGjOyImOulvCDrhbjroKTrnbwuIOq3uOuTpOydtCDsi5zsnqXsnYQg7Jew64ukLlxuXG4jIyA1LiDslrTrlrvqsowg7Y287KeA64KYIOKAlCDsiqTri4jsoIAoU25lZXplcinsmYAg7JWE7J2065SU7Ja0IOuwlOydtOufrOyKpFxuLSDsiqTri4jsoIAgPSDslYTsnbTrlJTslrTrpbwg7J6s7LGE6riw7ZWY65OvIO2NvOucqOumrOuKlCDsoITtjIzsnpAuIOyeheyGjOusuOydmCDtlbXsi6wuXG4tIOumrOuniOy7pOu4lO2VnCDqsoPrp4wg7KCE7YyM65Cc64ukLiDtj4nrspTtlZwg6rG0IOyVhOustOuPhCDsuZzqtazsl5Dqsowg66eQ7ZWY7KeAIOyViuuKlOuLpC5cbi0g6rSR6rOg67mEIOuMgOyLoCwg7Iqk64uI7KCA6rCAIOyekOuwnOyggeycvOuhnCDtjbzrnKjrprQg7J207Jyg66W8IOygnO2SiOyXkCDsi6zslrTrnbwuIO2NvOucqOumrOq4sCDsib3qsowg66eM65Ok7Ja06528LlxuXG4jIyA2LiDslrzrpqzslrTri7XthLDsmYAg7LqQ7KaYXG4tIOuMgOykkeydhCDsp4HsoJEg64W466as7KeAIOuniOudvC4g7Ja866as7Ja064u17YSw66W8IOuFuOumrOqzoCDqt7jrk6TsnbQg64uk7IiY7JeQ6rKMIOyghO2MjO2VmOqyjCDtlZjrnbwuXG4tIOumrOuniOy7pOu4lO2VmOyngCDslYrsnLzrqbQg7Ja866as7Ja064u17YSw7JmAIOuLpOyImCDsgqzsnbQg7LqQ7KaYKOqwhOq3uSnsnYQg66q7IOuEmOqzoCDsgqzrnbzsp4Tri6QuXG5cbiMjIDcuIOq3ueuLqOycvOuhnCwg6rCA7J6l7J6Q66as66GcXG4tIOyLnOyepSDtlZzqsIDsmrTrjbAo7KSR6rCEIO2SiOyniMK36rCA6rKpwrfrrLTrgpztlagp64qUIOqwgOyepSDrtpDruYTqs6Ag6rCA7J6lIOyViCDrs7Tsnbjri6QuXG4tIO2VnCDstpXsl5DshJwg6re564uo7Jy866GcOiDqsIDsnqUg67mg66W4L+yLvC/qs6DquIkv64uo7Iic7ZWcL+yghOusuOyggeyduC4g7Ja07KSR6rCE7ZWo7J20IOqwgOyepSDsnITtl5jtlZjri6QuXG5cbiMjIDguIOuRkOugpOybgOydtCDtj4nrspTtlajsnYQg66eM65Og64ukXG4tIOumrOuniOy7pOu4lO2VmOyngCDrqrvtlZjripQg7J207Jyg64qUIOu5hO2MkMK37Iuk7Yyo6rCAIOuRkOugpOybjOyEnC4g7JWI7KCEIn1dfQ=="
open("brain.jsonl", "w").write(base64.b64decode(_B64).decode("utf-8"))
ds = load_dataset("json", data_files="brain.jsonl", split="train")
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
def fmt(ex):
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False).removeprefix("<bos>") for c in ex["conversations"]]
    return {"text": texts}
ds = ds.map(fmt, batched=True)
print("데이터 개수:", len(ds)); print(ds[0]["text"][:400])


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:153: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


AttributeError: '_OpNamespace' '_c10d_functional' object has no attribute '_wrap_tensor_autograd'

In [12]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1, gradient_accumulation_steps = 4,
        warmup_steps = 5, max_steps = 40, learning_rate = 0.0003,
        logging_steps = 1, optim = "adamw_8bit", weight_decay = 0.001,
        lr_scheduler_type = "linear", seed = 3407, report_to = "none",
    ),
)


RuntimeError: Failed to import trl.trainer.sft_trainer because of the following error (look up to see its traceback):
Could not import module 'AutoProcessor'. Are this object's requirements defined correctly?

In [ ]:
# 🎭 응답(assistant)만 학습 — 질문 패턴은 마스킹(효율↑·품질↑)
# ⚠️ 마커는 모델/버전마다 다름(<|turn> vs <start_of_turn>) → 실제 텍스트에서 자동 감지
from unsloth.chat_templates import train_on_responses_only
_t = ds[0]["text"]
_im = "<|turn>user\n" if "<|turn>user" in _t else "<start_of_turn>user\n"
_rm = "<|turn>model\n" if "<|turn>model" in _t else "<start_of_turn>model\n"
trainer = train_on_responses_only(trainer, instruction_part=_im, response_part=_rm)
print(f"✅ 마스킹 마커 자동감지: {_rm.strip()} — 학습 준비 완료")


In [13]:
trainer_stats = trainer.train()
print("🎉 학습 완료! 최종 loss:", round(trainer_stats.training_loss, 4))
print("💡 loss 0.2~0.4면 sweet spot. 너무 낮으면(<0.1) 과적합 — max_steps 줄이세요.")


NameError: name 'trainer' is not defined

## 🧪 학습된 모델 테스트 (업로드 전에 확인!)
내가 가르친 지식을 직접 물어보세요. 답에 그 내용이 나오면 학습 성공이에요. 질문은 자유롭게 바꿔도 됩니다.


In [14]:
from unsloth import FastModel
FastModel.for_inference(model)
def chat(prompt, max_tokens=220):
    msg = [{"role":"user","content":[{"type":"text","text":prompt}]}]
    inp = tokenizer.apply_chat_template(msg, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to("cuda")
    if inp["input_ids"][0,0].item() == tokenizer.bos_token_id:
        inp["input_ids"] = inp["input_ids"][:,1:]; inp["attention_mask"] = inp["attention_mask"][:,1:]
    out = model.generate(**inp, max_new_tokens=max_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    ans = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\u2753 {prompt}\n\U0001F4AC {ans}\n" + "\u2500"*58)

# 👇 내가 가르친 지식에 대해 물어보세요 (자유롭게 수정)
chat("내 사업/지식에 대해 아는 걸 알려줘")
chat("너는 무엇을 도와줄 수 있어?")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:153: UserWarning: WARNING: Unsloth should be imported before [trl, transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


AttributeError: '_OpNamespace' '_c10d_functional' object has no attribute '_wrap_tensor_autograd'

## 💾 GGUF로 저장 (Connect AI 내장 엔진용)
테스트가 만족스러우면 업로드! (맨 앞에서 로그인했으니 바로 됩니다)


In [15]:
# 메모리 정리(OOM 방지) — 학습기 메모리 해제 후 변환
import gc, torch
try:
    del trainer
except Exception:
    pass
gc.collect(); torch.cuda.empty_cache()
print("메모리 정리 완료 — GGUF 변환 시작")
# 내 모델 = 장기 기억. q4_k_m GGUF 로 저장 + HF 업로드
model.push_to_hub_gguf("Marketing-ai-DX-v6", tokenizer, quantization_method="q4_k_m", token=True)
print("✅ 완료! Connect AI 앱 → 🤖 내 AI 팀 → HuggingFace에서 받기 → \"Marketing-ai-DX-v6\" 검색해서 내려받으면 내 모델로 바로 사용 (LM Studio 불필요)")


메모리 정리 완료 — GGUF 변환 시작


NameError: name 'model' is not defined